# 00 – Setup

## 1. Goal
Verify that the local environment is ready for the project: Python and package versions,
compute device (Apple Silicon GPU via MPS), project configuration, folder structure,
reproducibility (seeding) and the geodata stack (CRS EPSG:2056 / LV95).

## 2. Inputs / Outputs
| | |
| --- | --- |
| **Inputs** | `config.yaml`, `requirements.txt` |
| **Outputs** | `results/metrics/environment.json` (installed versions, device) |

## 3. Analysis
### 3.1 Project root and imports
All notebooks add the project root to `sys.path` so that the shared code in `src/` can be imported.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "config.yaml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import load_config, set_seed, get_device
print("Project root:", PROJECT_ROOT)

Project root: /Users/pascalingold/Dateien/HSLU/Computer Vision/Computer-Vision-Project


**Observation:** Path is correct, project folder is the Github-Repo.

### 3.2 Python and package versions

In [11]:
import platform
from importlib.metadata import version, PackageNotFoundError

packages = ["numpy", "pandas", "pyyaml", "geopandas", "shapely", "pyproj", "rasterio",
            "torch", "torchvision", "scikit-learn", "pillow", "matplotlib", "folium", "jupyterlab"]

versions = {"python": platform.python_version(), "platform": platform.platform()}
for p in packages:
    try:
        versions[p] = version(p)
    except PackageNotFoundError:
        versions[p] = "NOT INSTALLED"

for k, v in versions.items():
    print(f"{k:<14}{v}")

python        3.11.9
platform      macOS-27.2.1-arm64-arm-64bit
numpy         2.4.6
pandas        3.0.6
pyyaml        6.0.3
geopandas     1.2.0
shapely       2.2.0
pyproj        3.7.2
rasterio      1.4.4
torch         2.15.0
torchvision   0.29.2
scikit-learn  1.9.1
pillow        12.3.0
matplotlib    3.11.2
folium        0.20.0
jupyterlab    4.6.4


**Observation:** All necessary packets are loaded and their version number accords with the requirements.txt

### 3.3 Compute device
Without a cluster, all training runs locally. We check that PyTorch can use the Apple Silicon GPU (MPS)
and compare a matrix multiplication on CPU vs. MPS to get a feeling for the speed-up.

In [3]:
import time
import torch

cfg = load_config()
device = get_device(cfg["compute"]["device"])
print("Selected device:", device)
print("MPS available:  ", torch.backends.mps.is_available())
print("CUDA available: ", torch.cuda.is_available())

def bench(dev, n=4096, reps=10):
    a = torch.randn(n, n, device=dev)
    b = torch.randn(n, n, device=dev)
    (a @ b).sum().item()  # warm-up
    t0 = time.perf_counter()
    for _ in range(reps):
        c = a @ b
    c.sum().item()  # synchronise
    return (time.perf_counter() - t0) / reps

t_cpu = bench(torch.device("cpu"))
print(f"CPU: {t_cpu*1000:.1f} ms per 4096x4096 matmul")
if device.type != "cpu":
    t_dev = bench(device)
    print(f"{device.type.upper()}: {t_dev*1000:.1f} ms per 4096x4096 matmul  (speed-up x{t_cpu/t_dev:.1f})")

Selected device: mps
MPS available:   True
CUDA available:  False
CPU: 51.4 ms per 4096x4096 matmul
MPS: 25.2 ms per 4096x4096 matmul  (speed-up x2.0)


**Observation:** As expected, my Macbook has a MPS available and no CUDA. MPS is twice as fast as CPU. Local training has to be kept at a minimum, as large trainings won't be effective/doable.

### 3.4 Configuration and folder structure

In [4]:
import yaml

print(yaml.safe_dump({k: v for k, v in cfg.items() if k != "paths"}, sort_keys=False))
print("Paths:")
for name, p in cfg["paths"].items():
    print(f"  {name:<15}{p.relative_to(PROJECT_ROOT)!s:<20}{'ok' if p.exists() else 'MISSING'}")

project:
  name: solar-rooftop-detection-ch
  seed: 42
geo:
  crs: EPSG:2056
municipalities:
  uznach:
    name: Uznach
    canton: SG
    bfs_nr: 3339
  pully:
    name: Pully
    canton: VD
    bfs_nr: 5590
compute:
  device: auto
  num_workers: 4

Paths:
  data_raw       data/raw            ok
  data_interim   data/interim        ok
  data_processed data/processed      ok
  data_labels    data/labels         ok
  results        results             ok
  figures        results/figures     ok
  models         results/models      ok
  metrics        results/metrics     ok
  maps           results/maps        ok


**Observation:** YAML file is imported correctly, paths work.

### 3.5 Reproducibility
Seeding twice with the same seed must produce identical random numbers.

In [5]:
import numpy as np

set_seed(cfg["project"]["seed"])
a = (np.random.rand(3), torch.rand(3))
set_seed(cfg["project"]["seed"])
b = (np.random.rand(3), torch.rand(3))
print("NumPy identical:", np.allclose(a[0], b[0]))
print("Torch identical:", torch.equal(a[1], b[1]))

NumPy identical: True
Torch identical: True


**Observation:** Seeding works

### 3.6 Geodata stack
All geodata is handled in the Swiss coordinate system LV95 (EPSG:2056). As a check we transform the
approximate centres of both control municipalities from WGS84 to LV95 and print the GDAL version used by rasterio.

In [8]:
import rasterio
import geopandas as gpd
from shapely.geometry import Point

centres_wgs84 = {"Uznach": (8.983, 47.225), "Pully": (6.661, 46.510)}  # (lon, lat), approximate
gdf = gpd.GeoDataFrame(
    {"municipality": list(centres_wgs84)},
    geometry=[Point(xy) for xy in centres_wgs84.values()],
    crs="EPSG:4326",
).to_crs(cfg["geo"]["crs"])
gdf["E"] = gdf.geometry.x.round(0)
gdf["N"] = gdf.geometry.y.round(0)
print(gdf[["municipality", "E", "N"]].to_string(index=False))
print("\nGDAL version (rasterio):", rasterio.__gdal_version__)

municipality         E         N
      Uznach 2716959.0 1231603.0
       Pully 2540316.0 1151263.0

GDAL version (rasterio): 3.10.3


**Observation:** The coordinates are correct.

### 3.7 Save environment snapshot

In [9]:
import json

env = {**versions, "device": str(device)}
out_path = cfg["paths"]["metrics"] / "environment.json"
out_path.write_text(json.dumps(env, indent=2))
print("Written:", out_path.relative_to(PROJECT_ROOT))

Written: results/metrics/environment.json


## 4. Conclusion & Open Points
- The setup has worked. Paths are created and pushed to Github. Open Point is Training Capability and Local Power.